In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

WORK_DIR = '/content/drive/MyDrive/assistant-radio'
base = pd.read_csv(f'{WORK_DIR}/results_baseline.csv')
impr = pd.read_csv(f'{WORK_DIR}/results_improved.csv')

Mounted at /content/drive


In [2]:
import pandas as pd

def type_erreur(vrai, pred):
    """Categorise chaque cas selon la terminologie du bareme."""
    if pred == vrai:
        return "correct"
    if vrai == "suspected_opacity" and pred in ("normal", "uncertain"):
        return "faux negatif (opacite manquee)"
    if vrai == "normal" and pred == "suspected_opacity":
        return "faux positif"
    if pred == "uncertain":
        return "repli sur incertain"
    return "confusion de classe"

registre = pd.DataFrame({
    "cas": range(1, len(base) + 1),
    "fichier": base["filename"],
    "verite": base["true_class"],
    "pred_baseline": base["predicted_class"],
    "pred_improved": impr["predicted_class"],
    "conf_baseline": base["confidence"],
    "conf_improved": impr["confidence"],
})
registre["erreur_baseline"] = registre.apply(
    lambda r: type_erreur(r["verite"], r["pred_baseline"]), axis=1)
registre["erreur_improved"] = registre.apply(
    lambda r: type_erreur(r["verite"], r["pred_improved"]), axis=1)

# Sauvegarde dans le Drive (a cote des autres resultats)
registre.to_csv(f"{WORK_DIR}/registre_30_cas.csv", index=False)
print("Registre ecrit :", f"{WORK_DIR}/registre_30_cas.csv")

print("\n=== Repartition des types d'ecart ===")
synth = pd.DataFrame({
    "baseline": registre["erreur_baseline"].value_counts(),
    "improved": registre["erreur_improved"].value_counts(),
}).fillna(0).astype(int)
print(synth.to_string())

registre  # affiche le tableau complet des 30 cas

Registre ecrit : /content/drive/MyDrive/assistant-radio/registre_30_cas.csv

=== Repartition des types d'ecart ===
                                baseline  improved
confusion de classe                    7         7
correct                               15        12
faux negatif (opacite manquee)         7        10
faux positif                           1         0
repli sur incertain                    0         1


,cas,fichier,verite,pred_baseline,pred_improved,conf_baseline,conf_improved,erreur_baseline,erreur_improved
0,1,patient64576_view1_frontal.jpg,normal,normal,normal,0.95,0.80,correct,correct
1,2,patient64599_view1_frontal.jpg,normal,normal,normal,0.90,0.80,correct,correct
2,3,patient64544_view1_frontal.jpg,normal,normal,normal,0.95,0.80,correct,correct
3,4,patient64721_view1_frontal.jpg,normal,normal,normal,0.95,0.95,correct,correct
4,5,patient64588_view1_frontal.jpg,normal,normal,normal,0.95,0.80,correct,correct
5,6,patient64584_view1_frontal.jpg,normal,normal,normal,0.95,0.95,correct,correct
6,7,patient64593_view1_frontal.jpg,normal,normal,normal,0.95,0.80,correct,correct
7,8,patient64551_view1_frontal.jpg,normal,suspected_opacity,normal,0.60,0.80,faux positif,correct
8,9,patient64716_view1_frontal.jpg,normal,normal,uncertain,0.95,0.20,correct,repli sur incertain
9,10,patient64562_view1_frontal.jpg,normal,normal,normal,0.95,0.95,correct,correct


### Registre des 30 cas : analyse des erreurs

Le tableau ci-dessus catégorise chaque cas selon le type d'écart entre la vérité
terrain et la prédiction, pour les deux variantes.

Les **faux négatifs (opacités manquées) passent de 7 à 10**
entre baseline et improved. Autrement dit, le prompt « amélioré » manque
davantage d'anomalies réelles. C'est l'erreur la plus grave dans un contexte de
dépistage : une opacité non signalée peut correspondre à une pathologie non
détectée.

En parallèle, l'improved ne produit **aucun faux positif** (contre 1 pour la
baseline) — mais ce n'est pas une qualité : c'est la conséquence directe du fait
qu'il ne prédit plus jamais `suspected_opacity`. Le modèle a supprimé les faux
positifs en supprimant *toutes* les détections positives.

**Cas de réussite / d'échec à présenter en démo :**
- Réussite : les cas `normal` bien classés avec confiance élevée (~0.90).
- Échec caractéristique : cas 12 et 14, opacités réelles classées `normal` par
  l'improved — le sur-conservatisme transforme une détection en faux négatif.

Ce registre confirme le diagnostic des métriques : la règle de seuil trop
agressive déplace les erreurs vers le type le plus dangereux (faux négatifs).